<a href="https://colab.research.google.com/github/Vanquan99/Tool_Copy-Google-Drive/blob/main/tool_coppy_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder / File — Google Drive to Google Drive
Notebook dùng trên Colab để copy 1 file riêng lẻ hoặc copy toàn bộ folder giữa hai Google Drive (kể cả shared drive).

## Cách dùng nhanh
1. Mở notebook bằng Colab badge hoặc link GitHub.
2. Chạy lần lượt từng cell từ trên xuống dưới.
3. Dán link hoặc ID của folder/file nguồn và folder đích.
4. Nếu source là file, notebook sẽ copy trực tiếp file đó.

In [ ]:
#@title Input / Nhập thông tin
ui_language = "Tiếng Việt" #@param ["Tiếng Việt", "English"]
# Destination folder link / Link thư mục đích trên Google Drive
dest_link = "" #@param {type:"string"}
# Source file or folder link / Link file hoặc folder nguồn
source_link = "" #@param {type:"string"}
# Start page / Trang bắt đầu, set 0 to start from the first page
from_page = 0 #@param {type:"integer"}
# End page / Trang kết thúc, set 0 to include all pages
to_page = 0 #@param {type:"integer"}
# Maximum size in GB / Giới hạn dung lượng tối đa tính bằng GB
max_download_size_gb = 700 #@param {type:"number"}
# Excluded substrings, comma separated / Chuỗi loại trừ, ngăn cách bằng dấu phẩy
exclude_str = "" #@param {type:"string"}
# Show progress bars? / Hiển thị progress bar?
use_progress = True #@param [True, False]
# Leave pages at 0 to include everything.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
service = build('drive', 'v3')
print('Drive service ready')

In [ ]:
import time, re
from tqdm.notebook import tqdm

class DownloadFromDrive:
    def __init__(self, service, ui_language='vi', use_progress=True, log_file_path='copy_log.txt'):
        self.service = service
        self._total_size = 0.0  # MB
        self._limit_size = 0.0  # GB
        self.excluded_strings = []
        self.ui_language = 'vi' if str(ui_language).lower().startswith(('t', 'v')) else 'en'
        self.use_progress = use_progress
        self.log_file_path = log_file_path
        try:
            self.logfile = open(self.log_file_path, 'a', encoding='utf-8')
        except Exception:
            self.logfile = None

    MESSAGES = {
        'vi': {
            'start': 'Bắt đầu sao chép',
            'done': 'Hoàn tất',
            'skip_exist': 'Đã tồn tại, bỏ qua',
            'copying': 'Đang sao chép',
            'folder_enter': 'Vào thư mục',
            'done_run': 'Hoàn tất quá trình',
            'limit': 'Giới hạn (GB)',
            'exclude': 'Loại trừ'
        },
        'en': {
            'start': 'Starting copy',
            'done': 'Done',
            'skip_exist': 'Already exists, skipping',
            'copying': 'Copying',
            'folder_enter': 'Entering folder',
            'done_run': 'Run finished',
            'limit': 'Limit (GB)',
            'exclude': 'Exclude'
        }
    }

    def t(self, key):
        return self.MESSAGES.get(self.ui_language, self.MESSAGES['vi']).get(key, key)

    def log(self, message):
        ts = time.strftime('%Y-%m-%d %H:%M:%S')
        text = f'[{ts}] {message}'
        print(text)
        if self.logfile:
            try:
                self.logfile.write(text + '\n')
                self.logfile.flush()
            except Exception:
                pass

    def extract_id(self, value):
        if not value:
            return None
        match = re.search(r'[-\w]{25,}', value)
        return match.group(0) if match else value

    def is_folder(self, file_meta):
        return file_meta.get('mimeType') == 'application/vnd.google-apps.folder'

    def get_children(self, folder_id, from_page=0, to_page=0):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings:
            excluded_query = ' and '.join([f"not name contains '{item}'" for item in self.excluded_strings])
            query += f' and ({excluded_query})'
        page = 0
        while True:
            response = self.service.files().list(
                q=query,
                orderBy='name, createdTime',
                fields='files(id, name, mimeType, size), nextPageToken',
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            page += 1
            if (from_page < page <= to_page) or to_page == 0:
                files.extend(response.get('files', []))
            page_token = response.get('nextPageToken')
            if not page_token or (to_page > 0 and page >= to_page):
                break
        return files

    def exists_in_dest(self, dest_folder_id, name):
        try:
            escaped = name.replace("'", "\\'")
            query = f"'{dest_folder_id}' in parents and name = '{escaped}' and trashed = false"
            result = self.service.files().list(q=query, fields='files(id)').execute()
            return bool(result.get('files'))
        except Exception as exc:
            self.log(f"exists_in_dest error: {exc}")
            return False

    def create_folder(self, dest_folder_id, folder_name):
        if self.exists_in_dest(dest_folder_id, folder_name):
            result = self.service.files().list(
                q=f"'{dest_folder_id}' in parents and name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
                fields='files(id)'
            ).execute().get('files', [{}])
            return result[0].get('id') if result else None
        body = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}
        created = self.service.files().create(body=body, fields='id').execute()
        return created['id']

    def copy_file(self, dest_folder_id, file_meta):
        if self.exists_in_dest(dest_folder_id, file_meta['name']):
            self.log(f"{file_meta['name']} - {self.t('skip_exist')}")
            return
        start = time.time()
        self.service.files().copy(
            body={'parents': [dest_folder_id]},
            fileId=file_meta['id'],
            supportsAllDrives=True
        ).execute()
        elapsed = time.time() - start
        size_mb = int(file_meta.get('size', 0)) / (1024 * 1024) if file_meta.get('size') else 0.0
        self._total_size += size_mb
        self.log(f"[{file_meta['name']}] {self.t('copying')} {size_mb:0.2f} MB in {elapsed:0.2f}s")

    def copy_item(self, dest_folder_id, item_meta):
        if self.is_folder(item_meta):
            self.log(f"{self.t('folder_enter')}: {item_meta['name']}")
            new_folder_id = self.create_folder(dest_folder_id, item_meta['name'])
            children = self.get_children(item_meta['id'], 0, 0)
            self.copy_items(new_folder_id, children)
        else:
            self.copy_file(dest_folder_id, item_meta)

    def copy_items(self, dest_folder_id, items):
        iterator = tqdm(items, desc=self.t('copying')) if self.use_progress else items
        for item in iterator:
            self.copy_item(dest_folder_id, item)

    def copy_drive_to_drive(self, dest_link, source_link, from_page=0, to_page=0):
        dest_id = self.extract_id(dest_link)
        source_id = self.extract_id(source_link)
        if not dest_id or not source_id:
            raise ValueError('Missing destination or source ID')
        source_meta = self.service.files().get(fileId=source_id, supportsAllDrives=True, fields='id,name,mimeType,size').execute()
        if not self.is_folder(source_meta):
            self.log('Source is a file, copying directly')
            self.copy_file(dest_id, source_meta)
            return
        source_folder_name = source_meta['name']
        target_folder_id = self.create_folder(dest_id, source_folder_name)
        children = self.get_children(source_id, from_page, to_page)
        self.copy_items(target_folder_id, children)
        self.log(self.t('done'))

    def close(self):
        if getattr(self, 'logfile', None):
            try:
                self.logfile.close()
            except Exception:
                pass
            self.logfile = None

In [ ]:
downloader = DownloadFromDrive(service, ui_language=ui_language, use_progress=use_progress)
downloader._limit_size = float(max_download_size_gb)
downloader.excluded_strings = [item.strip() for item in exclude_str.split(',') if item.strip()]
downloader.log(downloader.t('start'))
try:
    downloader.copy_drive_to_drive(dest_link, source_link, from_page, to_page)
finally:
    downloader.log(downloader.t('done_run'))
    downloader.close()